### Load gold with weather

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "gold_nbhd_day_weather.parquet")
df["date"] = pd.to_datetime(df["date"])
print(df.shape)
df.head(2)

(692514, 22)


,AREA_ID,AREA_NAME,date,collisions,area_id,nbhd_id,area_name,ksi_collisions,ksi_fatal_collisions,ksi_serious_collisions,...,ksi_weighted_score,tavg,tmin,tmax,prcp,snow,wspd,freeze_day,snow_day,rain_day
0,2502366,South Eglinton-Davisville,2014-01-01,0,2502366,174,South Eglinton-Davisville,0,0,0,...,0,-11.1,-14.8,-8.8,0.0,0.21,18.0,1,1,0
1,2502366,South Eglinton-Davisville,2014-01-02,0,2502366,174,South Eglinton-Davisville,0,0,0,...,0,-17.3,-19.3,-14.9,2.5,2.24,23.1,1,1,1


### Sort and create lags/rolling for collisions and severity

In [2]:
# Identify your collision column
if "collisions" not in df.columns:
    # fallback: find likely name
    candidates = [c for c in df.columns if "collision" in c.lower()]
    raise ValueError(f"'collisions' not found. Candidates: {candidates}")

# KSI columns (only those that exist)
ksi_cols = [c for c in ["ksi_collisions","ksi_weighted_score","ksi_fatal_collisions","ksi_serious_collisions"] if c in df.columns]

df = df.sort_values(["nbhd_id","date"])

group = df.groupby("nbhd_id", group_keys=False)

# Lags
for L in [1, 7, 14]:
    df[f"collisions_lag{L}"] = group["collisions"].shift(L)
    for kc in ksi_cols:
        df[f"{kc}_lag{L}"] = group[kc].shift(L)

# Rolling means (use shift(1) before rolling to prevent leakage)
df["collisions_roll7_mean"]  = group["collisions"].shift(1).rolling(7).mean().reset_index(level=0, drop=True)
df["collisions_roll14_mean"] = group["collisions"].shift(1).rolling(14).mean().reset_index(level=0, drop=True)

for kc in ksi_cols:
    df[f"{kc}_roll7_mean"] = group[kc].shift(1).rolling(7).mean().reset_index(level=0, drop=True)

### Time features

In [3]:
df["dow"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

### Drop early rows where lags are NaN and keeping it simple for baseline

In [4]:
feature_cols = [c for c in df.columns if c.endswith(("lag1","lag7","lag14")) or "roll" in c or c in ["tavg","tmin","tmax","prcp","snow","wspd","freeze_day","snow_day","rain_day","dow","month"]]
needed = ["collisions_lag1","collisions_roll7_mean","tavg","prcp"]  # minimum sanity
for c in needed:
    if c not in df.columns:
        raise ValueError(f"Expected feature missing: {c}")

df_model = df.dropna(subset=["collisions_lag1","collisions_roll7_mean"]).copy()
print("After dropping lag NaNs:", df_model.shape)

After dropping lag NaNs: (691408, 46)


### Save feature table

In [5]:
out_path = PROCESSED_DIR / "features_nbhd_day.parquet"
df_model.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\processed\features_nbhd_day.parquet
